For modelling the penumbra we measure the penumbra width in line profile measurements in different field sizes. The penumbra measurements were taken at 10cm depth. 

Any other line profiles can also be used if the dose calculations are adjusted.

Our measurements are collected below.

| Field size    | MLC           | Jaw   |
| ------------- | ------------- | ---   |
| 5x5           | 4.733         | 4.039 |
| 10x10         | 5.226         | 4.550 |
| 15x15         | 5.495         | 4.823 |
| 20x20         | 5.947         | 5.111 |
| 30x30         | 6.492         | 5.592 |



In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import math
import pydicom
from pydose_rt import DoseEngine
from pydose_rt.data import MachineConfig, Phantom, loaders, Beam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype=torch.float32

do_plot = True

penumbra_fwhm_mlcs = [3.0]
penumbra_fwhm_jaws = [1.5]
field_sizes = [50, 100, 150, 200, 300]


resolution = (0.5, 2.0, 0.5)
ct_array_shape = (800, 100, 800)
phantom = Phantom.from_uniform_water(shape=ct_array_shape, spacing=resolution).to(device).to(dtype)

for penumbra_fwhm_mlc in penumbra_fwhm_mlcs:
    for penumbra_fwhm_jaw in penumbra_fwhm_jaws:
        results =  []
        for field_size in field_sizes:

            machine_config = MachineConfig(
                preset="../../src/pydose_rt/data/machine_presets/umea_10MV.json", 
                penumbra_fwhm=[penumbra_fwhm_mlc, penumbra_fwhm_jaw],
                head_scatter_amplitude=None, 
                head_scatter_sigma=None, 
                profile_corrections=None,
                output_factors=None
            )
            number_of_beams=1
            starting_angle=0
            iso_center=(200, 100, 200)
            kernel_size=401
            beam = Beam.create(
                gantry_angle_deg=0.0, 
                number_of_leaf_pairs=60, 
                collimator_angle_deg=0.0, 
                field_size_mm=(field_size, field_size), 
                iso_center=iso_center, 
                device=device, 
                dtype=dtype)
            dose_engine = DoseEngine(
                machine_config, 
                kernel_size,
                resolution=phantom._resolution,
                image_template=phantom.density_image,
                beam_template=beam,
                device=device,
                dtype=dtype,
                adjust_values=False
            )

            dose = dose_engine.compute_dose(
                beam,
                ct_image=phantom.density_image).cpu().detach().numpy()
            dose = dose[0, :, 50, :]
            mlc_profile = dose[400, :]
            mlc_20 = 0.2 * mlc_profile.max()
            mlc_80 = 0.8 * mlc_profile.max()
            jaw_profile = dose[:, 400]
            jaw_20 = 0.2 * jaw_profile.max()
            jaw_80 = 0.8 * jaw_profile.max()

            mlc_penumbra = (np.sum((mlc_profile > mlc_20) * (mlc_profile < mlc_80)) / 2) * resolution[0]
            jaw_penumbra = (np.sum((jaw_profile > jaw_20) * (jaw_profile < jaw_80)) / 2) * resolution[2]
            print(f"FS {field_size}x{field_size}: MLC: {mlc_penumbra} Jaw: {jaw_penumbra}")


FS 50x50: MLC: 5.0 Jaw: 4.5
FS 100x100: MLC: 5.5 Jaw: 4.5
FS 150x150: MLC: 6.0 Jaw: 5.0
FS 200x200: MLC: 6.0 Jaw: 5.0
FS 300x300: MLC: 6.0 Jaw: 5.0


: 